# 00 — Prepare the study data

This is the first of three notebooks. Here you build the dataset that notebooks 01 and 02 analyse.

## The causal question

> **To what extent does applying the backdoor defense, instead of baseline random filtering, change the probability of successful backdoor detection?**

| Variable | Meaning |
|---|---|
| `treatment = 0` | baseline random filtering |
| `treatment = 1` | backdoor defense applied |
| `outcome = 0` | detection failed |
| `outcome = 1` | detection succeeded |

`outcome` is binary, so its mean is the **detection success rate (DSR)**.

**Tutorial path:** **00 Data preparation** → 01 Correlational analysis → 02 Causal inference


## 1. What is real and what is synthetic

The raw CSV contains real code and docstring examples. They give the dataset realistic variation across units.

The treatment assignment and the detection outcome are **synthetic**: they are generated from a causal structure we control. This means:

- the numbers here are not empirical backdoor-defense results;
- because the true data-generating process is known, you can check your causal reasoning against it at the end of notebook 02.

Each row of the final table is one code example, observed under **one** treatment condition with **one** detection outcome.


## 2. Configure the preparation

Set the input file, the output paths, and the random seed used by this notebook.


In [ ]:
from src.causal_data_prep import (
    DEFAULT_COVARIATES,
    engineer_features,
    load_source_data,
    make_synthetic_observational_data,
    save_causal_dataset,
    save_ground_truth_dag,
    save_study_metadata,
    validate_causal_dataset,
)

def default_params():
    return {
        "source_dataset": "data/raw_code.csv",
        "lizard_cache_folder": "cache/lizard",
        "causal_dataset": "data/causal_data.csv",
        "ground_truth_dag": "data/synthetic_ground_truth_edges.csv",
        "study_metadata": "data/synthetic_study_metadata.json",
        "random_seed": 42,
        "covariate_columns": DEFAULT_COVARIATES,
    }

params = default_params()
params


## 3. Load the source examples

The source file contains:

| Column | Meaning |
|---|---|
| `input_code` | the code example |
| `output_docstring` | its docstring |
| `reviewer_experience` | synthetic pre-treatment context variable |
| `rollout_eligibility` | synthetic indicator of eligibility for the defense rollout |
| `noise_feature` | synthetic background variable with no intended meaning |

The last three are known **before** treatment is assigned. Keep that in mind — timing matters when you build the DAG.


In [ ]:
source_df = load_source_data(params["source_dataset"])

print(f"Loaded {len(source_df):,} source examples.")
source_df[
    [
        "input_code",
        "output_docstring",
        "reviewer_experience",
        "rollout_eligibility",
        "noise_feature",
    ]
].head()


## 4. Engineer code features

Derive four readable characteristics of each code example:

- `code_number_tokens`
- `code_complexity`
- `code_num_identifiers`
- `code_num_strings`

These are measurements only. A variable being present in the dataset says nothing about its causal role.


In [ ]:
feature_df = engineer_features(
    source_df,
    cache_dir=params["lizard_cache_folder"],
)

feature_df[
    [
        "code_number_tokens",
        "code_complexity",
        "code_num_identifiers",
        "code_num_strings",
        "reviewer_experience",
        "rollout_eligibility",
        "noise_feature",
    ]
].describe().T


## 5. Generate the observed treatment and outcome

Each unit is assigned to exactly one condition:

- `0` — random filtering (the baseline);
- `1` — backdoor defense applied.

`outcome` records whether detection succeeded for that unit.

Two variables, `inspection_intensity` and `manual_review_flag`, are measured **after** treatment. They are included deliberately: both correlate strongly with treatment and outcome, which makes the DAG exercise realistic. Strong association does not make a variable a confounder.

The generator knows both potential outcomes for every unit, but the exported dataset keeps only the one actually observed.


In [ ]:
causal_df, study_info = make_synthetic_observational_data(
    feature_df,
    seed=params["random_seed"],
)

print(f"Backdoor-defense prevalence: {study_info.treatment_prevalence:.3f}")
print(f"Observed detection success rate: {study_info.outcome_prevalence:.3f}")

causal_df.head()


## 6. Validate the table

Check that:

- every unit appears exactly once;
- both treatment conditions are present;
- `outcome` is binary;
- the analysis variables are numeric and finite;
- no analysis values are missing.


In [ ]:
validation_summary = validate_causal_dataset(
    causal_df,
    covariates=params["covariate_columns"],
)

validation_summary


## 7. Save the outputs

This cell writes three files:

| File | Used by |
|---|---|
| `data/causal_data.csv` | notebooks 01 and 02 |
| `data/synthetic_ground_truth_edges.csv` | the reveal section of notebook 02 |
| `data/synthetic_study_metadata.json` | the reveal section of notebook 02 |

The last two hold the true DAG and the true ATE. **Do not open them yet** — you will compare them against your own answer at the end of notebook 02.


In [ ]:
data_path = save_causal_dataset(
    causal_df,
    params["causal_dataset"],
)
truth_path = save_ground_truth_dag(
    params["ground_truth_dag"],
)
metadata_path = save_study_metadata(
    study_info,
    params["study_metadata"],
)

print(f"Saved causal dataset: {data_path}")
print(f"Saved hidden DAG: {truth_path}")
print(f"Saved hidden study metadata: {metadata_path}")


## Next: notebook 01

Notebook 00 answered: **what was observed for each unit?**

Notebook 01 sets the hidden DAG aside and asks: **what relationships can we see in the observed data?**

That is a correlational question, not yet a causal one.
